# Paper 2 — High-Impact Detector-Conflict Analysis

This notebook analyzes the **78 high-impact detector-conflict files** described in Paper 2. It is designed for Google Colab/Google Drive, but it can also run locally when the exported project files are placed beside the notebook.

**High-impact conflict definition:** a file is flagged when:

1. at least one available detector score is **≤ 30% AI**,
2. at least one available detector score is **≥ 70% AI**, and
3. the **Combined AI rate** is between **40% and 60%** inclusive.

The notebook compares high-impact conflict files with all other files by language, assignment type, major, university, year grouping, word count, detector-score range, detector-score variance, and detector-specific disagreement patterns.

Optional OpenAI-assisted qualitative coding is included, but it is **off by default**. If enabled, the notebook sends only de-identified excerpts or derived features and uses a strict JSON schema. No result should be interpreted as proof that any individual file was AI-generated.

In [ ]:
# =========================================================
# 0. Setup
# =========================================================
import re
import sys
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from IPython.display import display

import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 180)

try:
    from docx import Document

    DOCX_AVAILABLE = True
except Exception:
    DOCX_AVAILABLE = False

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

OUTPUT_DIR = (
    Path("${PROJECT_DATA_PATH}")
    if IN_COLAB
    else Path("Paper2_High_Impact_Conflict_Analysis_Outputs")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Outputs will be saved to:", OUTPUT_DIR.resolve())

## 1. Load assignment-level detector results

The expected dataset is the combined primary-sample detector file, usually named `AI_Detection_Results_Combined.xlsx` or `AI_Detection_Results_Combined.csv` under an `Assignments Part/CSV Results` folder. The notebook tries several paths learned from the earlier project notebooks and ZIP structure.

In [ ]:
# =========================================================
# 1. Data path candidates learned from earlier notebooks and ZIP structure
# =========================================================
PATH_CANDIDATES = [
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("/content/AI_Detection_Results_Combined.xlsx"),
    Path("/content/AI_Detection_Results_Combined.csv"),
    Path("AI_Detection_Results_Combined.xlsx"),
    Path("AI_Detection_Results_Combined.csv"),
    # local validation path from the uploaded ZIP extraction
    Path("/mnt/data/paper2_src/CSV Results/AI_Detection_Results_Combined.xlsx"),
]


def find_first_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None


DATA_PATH = find_first_existing_path(PATH_CANDIDATES)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files

    print(
        "No data file found in path candidates. Please upload AI_Detection_Results_Combined.xlsx or CSV."
    )
    uploaded = files.upload()
    DATA_PATH = Path(next(iter(uploaded.keys())))

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find AI_Detection_Results_Combined.xlsx/csv. Update PATH_CANDIDATES or upload the file."
    )

print("Using data file:", DATA_PATH)
if str(DATA_PATH).lower().endswith((".xlsx", ".xls")):
    raw = pd.read_excel(DATA_PATH)
else:
    raw = pd.read_csv(DATA_PATH)

print("Raw shape:", raw.shape)
display(raw.head())

In [ ]:
def normalize_colname(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[%()]+", "", column_name)
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    return re.sub(r"_+", "_", column_name).strip("_")


raw_to_norm = {column: normalize_colname(column) for column in raw.columns}
df = raw.rename(columns=raw_to_norm).copy()

aliases = {
    "language": "language",
    "university": "university",
    "major": "major",
    "assignment_type": "assignment_type",
    "assignmenttype": "assignment_type",
    "year": "year_grouping",
    "year_group": "year_grouping",
    "year_grouping": "year_grouping",
    "file_name": "file_name",
    "filename": "file_name",
    "file_path": "file_path",
    "path": "file_path",
    "word_count": "word_count",
    "words": "word_count",
    "weighted_ai_rate": "combined_ai_rate_100",
    "combined_ai_rate": "combined_ai_rate_100",
    "combined_ai_rate_percent": "combined_ai_rate_100",
    "combined_ai_score": "combined_ai_rate_100",
    "combined_score": "combined_ai_rate_100",
    "weighted_result": "combined_result",
    "gptzero_ai_rate_percent": "gptzero_ai_rate_percent",
    "gptzero_score_percent": "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent": "pangram_ai_rate_percent",
    "pangram_score_percent": "pangram_ai_rate_percent",
    "sapling_ai_rate_percent": "sapling_ai_rate_percent",
    "sapling_score_percent": "sapling_ai_rate_percent",
    "isgen_ai_rate_percent": "isgen_ai_rate_percent",
    "isgen_score_percent": "isgen_ai_rate_percent",
}
df = df.rename(
    columns={column: aliases[column] for column in df.columns if column in aliases}
)

required = [
    "combined_ai_rate_100",
    "language",
    "assignment_type",
    "word_count",
    "major",
    "university",
    "year_grouping",
    "file_name",
]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns after standardization: {missing}. "
        f"Available columns: {df.columns.tolist()}"
    )

score_cols = [
    column
    for column in [
        "gptzero_ai_rate_percent",
        "pangram_ai_rate_percent",
        "sapling_ai_rate_percent",
        "isgen_ai_rate_percent",
    ]
    if column in df.columns
]
if len(score_cols) < 2:
    raise ValueError(f"Expected detector score columns. Found: {score_cols}")

for column in ["combined_ai_rate_100", "word_count", *score_cols]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in [
    "language",
    "assignment_type",
    "major",
    "university",
    "year_grouping",
    "file_name",
]:
    df[column] = df[column].astype(str).str.strip()
    df.loc[df[column].isin(["", "nan", "None", "<NA>"]), column] = np.nan

# Detector outcomes are modeled as fractions while retaining the original percent scale.
if df["combined_ai_rate_100"].max(skipna=True) <= 1.000001:
    df["combined_ai_rate_100"] = df["combined_ai_rate_100"] * 100

df["combined_ai_rate_01"] = (df["combined_ai_rate_100"] / 100).clip(0, 1)
df["log_word_count"] = np.log(df["word_count"].clip(lower=1))

# File names may not uniquely identify records.
df = df.reset_index(drop=True)
df.insert(0, "row_id", np.arange(1, len(df) + 1))

## 2. Flag high-impact detector conflicts

This section implements the paper definition exactly. Code files can have fewer available detector scores because they were scored differently; the high-impact conflict flag is computed using available detector score columns and remains cautious.

In [ ]:
# =========================================================
# 3. Detector disagreement and high-impact conflict flag
# Compute per-row detector summary statistics and mark assignments where
# detectors strongly disagree and the aggregated score is near the midpoint.
# "High-impact conflict": at least one detector <=30, at least one >=70,
# and combined_ai_rate_100 in [40, 60].
# =========================================================
score_matrix = df[score_cols]

# Basic per-row summary statistics across available detector scores
df["detector_score_min"] = score_matrix.min(axis=1, skipna=True)
df["detector_score_max"] = score_matrix.max(axis=1, skipna=True)
df["detector_score_range"] = df["detector_score_max"] - df["detector_score_min"]
df["detector_score_mean"] = score_matrix.mean(axis=1, skipna=True)
df["detector_score_variance"] = score_matrix.var(axis=1, skipna=True)
df["n_detector_scores_available"] = score_matrix.notna().sum(axis=1)

# Logical flags for extreme/ambiguous detector outputs and combined rate window
df["any_detector_le_30"] = (score_matrix <= 30).any(axis=1)
df["any_detector_ge_70"] = (score_matrix >= 70).any(axis=1)
# combined_ai_rate_100 near midpoint suggests overall ambiguity; include both endpoints
df["combined_mid_40_60"] = df["combined_ai_rate_100"].between(40, 60, inclusive="both")

# Mark rows where detectors conflict (low vs high) and the combined score is ambiguous
df["high_impact_conflict"] = (
    df["any_detector_le_30"] & df["any_detector_ge_70"] & df["combined_mid_40_60"]
)

# Per-detector indicator columns for downstream filtering/inspection
for c in score_cols:
    tool = c.replace("_ai_rate_percent", "")
    df[f"{tool}_le_30"] = df[c] <= 30
    df[f"{tool}_ge_70"] = df[c] >= 70
    df[f"{tool}_mid_40_60"] = df[c].between(40, 60, inclusive="both")

print("Total files:", len(df))
print("High-impact conflict files:", int(df["high_impact_conflict"].sum()))
print("Non-high-impact conflict files:", int((~df["high_impact_conflict"]).sum()))

display(
    df.loc[
        df["high_impact_conflict"],
        [
            "row_id",
            "file_name",
            "language",
            "assignment_type",
            "major",
            "university",
            "year_grouping",
            "combined_ai_rate_100",
            "detector_score_min",
            "detector_score_max",
            "detector_score_range",
        ]
        + score_cols,
    ].head(10)
)

# Export flagged rows for reporting. CSV uses UTF-8 with BOM to preserve encoding in Excel.
flagged_out = OUTPUT_DIR / "paper2_assignment_level_high_impact_conflict_flags.csv"
df.to_csv(flagged_out, index=False, encoding="utf-8-sig")
print("Saved:", flagged_out)

## 3. Resolve cleaned text file paths

The combined primary-sample results file may not contain a `file_path` column. This section tries to find cleaned text files using known Google Drive roots from the earlier notebooks. If paths are not found, numeric conflict analysis still runs; OpenAI-assisted textual-feature coding will be skipped until the text paths are available.

In [ ]:
# =========================================================
# 4. Resolve cleaned text file paths by file_name, if needed
# =========================================================
TEXT_ROOT_CANDIDATES = [
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("/mnt/data/paper2_src"),
    Path("."),
]


def build_text_file_index(roots):
    mapping = defaultdict(list)
    scanned_roots = []
    for root in roots:
        if not root.exists():
            continue
        scanned_roots.append(str(root))
        # Avoid scanning extremely large roots unless they are specific.
        for p in root.rglob("*.txt"):
            mapping[p.name].append(str(p))
    return mapping, scanned_roots


# If the dataset already has a path column, preserve it.
if "file_path" in df.columns:
    df["file_path_resolved"] = df["file_path"].astype(str)
    df.loc[
        df["file_path_resolved"].isin(["", "nan", "None", "<NA>"]), "file_path_resolved"
    ] = np.nan
else:
    df["file_path_resolved"] = np.nan

need_path = df["file_path_resolved"].isna().any()
text_index, scanned_roots = (
    build_text_file_index(TEXT_ROOT_CANDIDATES) if need_path else ({}, [])
)
print("Scanned text roots:", scanned_roots)
print("Indexed text file names:", len(text_index))


def choose_best_path(row, candidates):
    if not candidates:
        return np.nan
    if len(candidates) == 1:
        return candidates[0]
    # Prefer paths containing the categorical labels when possible.
    tokens = [
        row.get("university", ""),
        row.get("year_grouping", ""),
        row.get("assignment_type", ""),
        row.get("major", ""),
        row.get("language", ""),
    ]
    tokens = [str(t).lower().replace(" ", "") for t in tokens if pd.notna(t)]
    scored = []
    for p in candidates:
        s = p.lower().replace(" ", "")
        score = sum(tok in s for tok in tokens if tok)
        scored.append((score, len(p), p))
    scored.sort(reverse=True)
    return scored[0][2]


if need_path:
    resolved = []
    n_duplicates = 0
    for _, row in df.iterrows():
        fname = str(row["file_name"])
        candidates = text_index.get(fname, [])
        if len(candidates) > 1:
            n_duplicates += 1
        resolved.append(choose_best_path(row, candidates))
    df["file_path_resolved"] = df["file_path_resolved"].fillna(
        pd.Series(resolved, index=df.index)
    )
    print("Rows with duplicate path candidates:", n_duplicates)

# Verify file existence


def path_exists(x):
    try:
        return Path(str(x)).exists()
    except Exception:
        return False


df["file_path_exists"] = df["file_path_resolved"].apply(path_exists)
print("Resolved existing file paths:", int(df["file_path_exists"].sum()), "of", len(df))
print(
    "Resolved existing paths among high-impact conflicts:",
    int(df.loc[df["high_impact_conflict"], "file_path_exists"].sum()),
    "of",
    int(df["high_impact_conflict"].sum()),
)

display(df[["row_id", "file_name", "file_path_resolved", "file_path_exists"]].head())

## 4. Compare high-conflict and non-high-conflict files

The comparisons below are descriptive and exploratory. They identify corpus patterns associated with high-impact detector disagreement; they do not determine whether any individual file was AI-generated.

In [ ]:
# =========================================================
# 5. Categorical comparisons: language, assignment type, major, university, year
# =========================================================
def cramer_v_from_table(ct):
    chi2, p, dof, expected = chi2_contingency(ct)
    n = ct.to_numpy().sum()
    if n <= 1:
        return np.nan, p
    r, k = ct.shape
    phi2 = chi2 / n
    # Apply bias correction for small samples/categories (Bergsma & Wicher adaptation)
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    v = np.sqrt(phi2corr / denom) if denom > 0 else np.nan
    return v, p


cat_vars = ["language", "assignment_type", "major", "university", "year_grouping"]
cat_summary_rows = []
cat_crosstabs = {}

for var in cat_vars:
    ct = pd.crosstab(df[var], df["high_impact_conflict"])
    # Ensure both boolean outcome columns exist before downstream calculations
    ct = ct.reindex(columns=[False, True], fill_value=0)
    ct.columns = ["non_high_conflict_n", "high_conflict_n"]
    ct["total_n"] = ct.sum(axis=1)
    # percent of cases within each category labeled as high impact conflict
    ct["high_conflict_percent_within_category"] = (
        100 * ct["high_conflict_n"] / ct["total_n"]
    ).round(1)
    cat_crosstabs[var] = ct.reset_index()
    v, p = cramer_v_from_table(pd.crosstab(df[var], df["high_impact_conflict"]))
    cat_summary_rows.append(
        {
            "variable": var,
            "cramers_v": v,
            "chi_square_p": p,
            "n_categories": df[var].nunique(dropna=False),
        }
    )
    print("\n" + str(var) + " by high-impact conflict")
    display(ct.sort_values("high_conflict_percent_within_category", ascending=False))

cat_test_summary = pd.DataFrame(cat_summary_rows)
display(cat_test_summary)

# Export summary and detailed crosstabs for reporting. Sheet names truncated to Excel limit.
cat_test_summary.to_csv(
    TABLE_DIR / "categorical_association_tests.csv", index=False, encoding="utf-8-sig"
)
with pd.ExcelWriter(
    TABLE_DIR / "categorical_crosstabs_by_high_conflict.xlsx", engine="openpyxl"
) as writer:
    cat_test_summary.to_excel(writer, sheet_name="association_tests", index=False)
    for var, table in cat_crosstabs.items():
        table.to_excel(writer, sheet_name=var[:31], index=False)

In [ ]:
# =========================================================
# 6. Continuous comparisons: word count and detector disagreement metrics
# =========================================================
def cliffs_delta(x, y):
    x = np.asarray(pd.Series(x).dropna())
    y = np.asarray(pd.Series(y).dropna())
    if len(x) == 0 or len(y) == 0:
        return np.nan
    # Efficient enough for n=543
    gt = sum((xi > y).sum() for xi in x)
    lt = sum((xi < y).sum() for xi in x)
    return (gt - lt) / (len(x) * len(y))


continuous_vars = [
    "word_count",
    "log_word_count",
    "combined_ai_rate_100",
    "detector_score_range",
    "detector_score_variance",
    "detector_score_min",
    "detector_score_max",
] + score_cols
cont_rows = []
for var in continuous_vars:
    hi = df.loc[df["high_impact_conflict"], var].dropna()
    non = df.loc[~df["high_impact_conflict"], var].dropna()
    try:
        u_stat, u_p = (
            mannwhitneyu(hi, non, alternative="two-sided")
            if len(hi) and len(non)
            else (np.nan, np.nan)
        )
    except Exception:
        u_stat, u_p = np.nan, np.nan
    try:
        t_stat, t_p = (
            ttest_ind(hi, non, equal_var=False, nan_policy="omit")
            if len(hi) and len(non)
            else (np.nan, np.nan)
        )
    except Exception:
        t_stat, t_p = np.nan, np.nan
    cont_rows.append(
        {
            "variable": var,
            "high_conflict_n": len(hi),
            "high_conflict_mean": hi.mean(),
            "high_conflict_sd": hi.std(),
            "high_conflict_median": hi.median(),
            "non_high_conflict_n": len(non),
            "non_high_conflict_mean": non.mean(),
            "non_high_conflict_sd": non.std(),
            "non_high_conflict_median": non.median(),
            "mean_difference_high_minus_non": hi.mean() - non.mean(),
            "mann_whitney_p": u_p,
            "welch_t_p": t_p,
            "cliffs_delta": cliffs_delta(hi, non),
        }
    )

continuous_comparison = pd.DataFrame(cont_rows)
display(continuous_comparison)
continuous_comparison.to_csv(
    TABLE_DIR / "continuous_comparisons_high_vs_non_high_conflict.csv",
    index=False,
    encoding="utf-8-sig",
)

## 5. Detector-specific disagreement patterns

These tables show which tools tend to be the low-scoring detector, high-scoring detector, or part of the low/high split in high-impact conflict files.

In [ ]:
# =========================================================
# 7. Detector-specific patterns
# =========================================================
def detector_pattern(row):
    low_tools = []
    high_tools = []
    mid_tools = []
    for c in score_cols:
        tool = c.replace("_ai_rate_percent", "")
        val = row[c]
        if pd.isna(val):
            continue
        if val <= 30:
            low_tools.append(tool)
        if val >= 70:
            high_tools.append(tool)
        if 40 <= val <= 60:
            mid_tools.append(tool)
    return pd.Series(
        {
            "low_tools_le_30": ", ".join(low_tools) if low_tools else "None",
            "high_tools_ge_70": ", ".join(high_tools) if high_tools else "None",
            "mid_tools_40_60": ", ".join(mid_tools) if mid_tools else "None",
            "n_low_tools": len(low_tools),
            "n_high_tools": len(high_tools),
            "n_mid_tools": len(mid_tools),
            "low_high_pattern": f"LOW: {'+'.join(low_tools) if low_tools else 'None'} | HIGH: {'+'.join(high_tools) if high_tools else 'None'}",
        }
    )


pattern_df = df.apply(detector_pattern, axis=1)
df = pd.concat([df, pattern_df], axis=1)

pattern_counts = (
    df.loc[df["high_impact_conflict"], "low_high_pattern"]
    .value_counts()
    .rename_axis("low_high_pattern")
    .reset_index(name="high_conflict_n")
)
pattern_counts["percent_of_high_conflicts"] = (
    100 * pattern_counts["high_conflict_n"] / df["high_impact_conflict"].sum()
).round(1)
print("Most common low/high detector patterns among high-impact conflicts")
display(pattern_counts.head(20))

# Tool-level low/high participation
participation_rows = []
for c in score_cols:
    tool = c.replace("_ai_rate_percent", "")
    for group_value, group_label in [
        (True, "high_conflict"),
        (False, "non_high_conflict"),
    ]:
        sub = df[df["high_impact_conflict"] == group_value]
        participation_rows.append(
            {
                "tool": tool,
                "group": group_label,
                "n_available": int(sub[c].notna().sum()),
                "n_le_30": int((sub[c] <= 30).sum()),
                "percent_le_30": 100
                * (sub[c] <= 30).sum()
                / max(sub[c].notna().sum(), 1),
                "n_ge_70": int((sub[c] >= 70).sum()),
                "percent_ge_70": 100
                * (sub[c] >= 70).sum()
                / max(sub[c].notna().sum(), 1),
                "mean_score": sub[c].mean(),
                "median_score": sub[c].median(),
            }
        )

tool_participation = pd.DataFrame(participation_rows)
display(tool_participation)

pattern_counts.to_csv(
    TABLE_DIR / "high_conflict_detector_low_high_patterns.csv",
    index=False,
    encoding="utf-8-sig",
)
tool_participation.to_csv(
    TABLE_DIR / "detector_specific_low_high_participation.csv",
    index=False,
    encoding="utf-8-sig",
)

## 6. Charts

Charts are saved as PNG files for manuscript or appendix use. They are descriptive and should be interpreted at the corpus level only.

In [ ]:
# =========================================================
# 8. Charts
# =========================================================
def save_bar_chart(series, title, xlabel, ylabel, filename):
    ax = series.plot(kind="bar", figsize=(8, 4))
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    out = FIG_DIR / filename
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    return out


# Plot percentage of high-impact conflicts for selected categorical variables
for var in ["language", "assignment_type", "major", "university", "year_grouping"]:
    tab = cat_crosstabs[var].copy()
    # sort categories by descending high-conflict percentage
    tab = tab.sort_values("high_conflict_percent_within_category", ascending=False)
    ser = tab.set_index(var)["high_conflict_percent_within_category"]
    fname = f"high_conflict_percent_by_{var}.png"
    save_bar_chart(
        ser,
        f"High-impact conflict percentage by {var.replace('_', ' ')}",
        var.replace("_", " "),
        "High-impact conflict (%)",
        fname,
    )

# Boxplots comparing distributions of numeric measures by conflict status
for var in [
    "word_count",
    "detector_score_range",
    "detector_score_variance",
    "combined_ai_rate_100",
]:
    fig, ax = plt.subplots(figsize=(6, 4))
    # separate values for non-high and high-impact conflict groups, drop missing
    data = [
        df.loc[~df["high_impact_conflict"], var].dropna(),
        df.loc[df["high_impact_conflict"], var].dropna(),
    ]
    ax.boxplot(data, labels=["Non-high-conflict", "High-conflict"], showmeans=True)
    ax.set_title(f"{var.replace('_', ' ').title()} by high-impact conflict status")
    ax.set_ylabel(var.replace("_", " "))
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    out = FIG_DIR / f"boxplot_{var}_by_high_conflict.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()

print("Saved charts to:", FIG_DIR.resolve())

## 7. Optional OpenAI-assisted textual feature coding

This section is optional and is **off by default**. It is included because the feedback asks for a structured review of likely textual/formatting reasons detectors disagree.

Privacy safeguards:

- only de-identified excerpts or derived text features are sent;
- file names, student names, university names, and obvious IDs are removed from the prompt;
- the model is asked to identify **textual features**, not authorship;
- outputs must not be used to accuse any student or determine whether a file is AI-generated.

Set `RUN_OPENAI_FEATURE_CODING = True` only after confirming that the research ethics/consent terms allow this use.

In [ ]:
# =========================================================
# 9. Optional de-identified textual feature coding with strict JSON schema
# =========================================================
RUN_OPENAI_FEATURE_CODING = False  # Change to True only if permitted by ethics/consent and OpenAI API access is available.
OPENAI_MODEL = "gpt-4.1-mini"  # adjust if needed
MAX_EXCERPT_CHARS = 4500
MAX_FILES_TO_CODE = (
    None  # None = all files with available text paths; or set e.g., 120 for a pilot
)

FEATURE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "template_artifacts",
        "mixed_writing_style",
        "reference_heavy_text",
        "formulaic_structure",
        "non_native_english_signals",
        "technical_jargon",
        "short_or_long_length",
        "paraphrase_like_phrasing",
        "formatting_artifacts",
        "language_or_translation_mixture",
        "likely_conflict_reasons",
        "confidence",
    ],
    "properties": {
        "template_artifacts": {"type": "boolean"},
        "mixed_writing_style": {"type": "boolean"},
        "reference_heavy_text": {"type": "boolean"},
        "formulaic_structure": {"type": "boolean"},
        "non_native_english_signals": {"type": "boolean"},
        "technical_jargon": {"type": "boolean"},
        "short_or_long_length": {
            "type": "string",
            "enum": ["short", "typical", "long", "unclear"],
        },
        "paraphrase_like_phrasing": {"type": "boolean"},
        "formatting_artifacts": {"type": "boolean"},
        "language_or_translation_mixture": {"type": "boolean"},
        "likely_conflict_reasons": {
            "type": "array",
            "minItems": 1,
            "maxItems": 5,
            "items": {"type": "string"},
        },
        "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
    },
}

IDENTIFIER_PATTERNS = [
    r"[A-Z][a-z]+\s+[A-Z][a-z]+",  # simple English full names
    r"\d{6,}",  # long IDs
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    r"(?:Taif|Taibah|KFUPM|Jeddah|King Fahd|University of Jeddah)",
    r"(?:Dr\.|Doctor|Professor|Prof\.)\s+[A-Z][A-Za-z]+",
]


def read_text_file(path):
    encodings = ["utf-8", "utf-8-sig", "cp1256", "cp1252", "latin-1"]
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="strict") as f:
                return f.read()
        except Exception:
            continue
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def deidentify_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    for pat in IDENTIFIER_PATTERNS:
        text = re.sub(pat, "[REDACTED]", text, flags=re.IGNORECASE)
    # Remove repeated whitespace but preserve paragraph breaks
    text = re.sub(r"[ 	]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def build_excerpt(text, max_chars=MAX_EXCERPT_CHARS):
    text = deidentify_text(text)
    if len(text) <= max_chars:
        return text
    first = text[: max_chars // 2]
    last = text[-max_chars // 2 :]
    return first + "\n\n[...middle omitted for privacy and length...]\n\n" + last


feature_rows = []

if RUN_OPENAI_FEATURE_CODING:
    try:
        from openai import OpenAI
    except Exception:
        raise ImportError("Install the OpenAI package first: pip install openai")
    client = OpenAI()

    candidates = df[df["file_path_exists"]].copy()
    # Prioritize high-impact conflicts, then sample non-high-conflicts for comparison.
    candidates = pd.concat(
        [
            candidates[candidates["high_impact_conflict"]].sort_values(
                "detector_score_range", ascending=False
            ),
            candidates[~candidates["high_impact_conflict"]].sample(
                min(80, (~candidates["high_impact_conflict"]).sum()), random_state=42
            ),
        ],
        ignore_index=True,
    )
    if MAX_FILES_TO_CODE is not None:
        candidates = candidates.head(MAX_FILES_TO_CODE)

    for i, row in candidates.iterrows():
        path = row["file_path_resolved"]
        try:
            raw_text = read_text_file(path)
            excerpt = build_excerpt(raw_text)
            prompt = f"""
You are coding de-identified student-assignment excerpts for corpus-level research on detector disagreement.
Do not infer or state whether the file is AI-generated. Code only observable textual or formatting features that could plausibly affect detector disagreement.

Context variables:
- language: {row['language']}
- assignment_type: {row['assignment_type']}
- word_count: {row['word_count']}
- detector_score_range: {row['detector_score_range']:.2f}

De-identified excerpt:
{excerpt}
""".strip()
            resp = client.responses.create(
                model=OPENAI_MODEL,
                input=[{"role": "user", "content": prompt}],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "detector_conflict_text_features",
                        "schema": FEATURE_SCHEMA,
                        "strict": True,
                    }
                },
            )
            # The SDK returns parsed text; parse defensively.
            txt = resp.output_text
            coded = json.loads(txt)
            feature_rows.append(
                {
                    "row_id": row["row_id"],
                    "file_name": row["file_name"],
                    "high_impact_conflict": row["high_impact_conflict"],
                    **coded,
                }
            )
            if (i + 1) % 10 == 0:
                print("Coded", i + 1, "files")
        except Exception as e:
            feature_rows.append(
                {
                    "row_id": row["row_id"],
                    "file_name": row["file_name"],
                    "high_impact_conflict": row["high_impact_conflict"],
                    "coding_error": str(e),
                }
            )

    feature_codes = pd.DataFrame(feature_rows)
    feature_codes.to_csv(
        TABLE_DIR / "openai_textual_feature_codes.csv",
        index=False,
        encoding="utf-8-sig",
    )
    display(feature_codes.head())
else:
    print(
        "OpenAI feature coding is OFF. Set RUN_OPENAI_FEATURE_CODING = True to run it after ethics/consent review."
    )
    feature_codes = pd.DataFrame()

## 8. Manual validation sample

This creates a review sheet for manual validation. The goal is to validate whether the textual-feature codes are reasonable, not to classify authorship. Reviewers should not attempt to identify students or infer whether any file was AI-generated.

In [ ]:
# =========================================================
# 10. Manual validation sample
# =========================================================
VALIDATION_HIGH_N = 10
VALIDATION_NON_N = 10

validation_pool = df.copy()
hi_sample = validation_pool[validation_pool["high_impact_conflict"]].sample(
    min(VALIDATION_HIGH_N, validation_pool["high_impact_conflict"].sum()),
    random_state=123,
)
non_sample = validation_pool[~validation_pool["high_impact_conflict"]].sample(
    min(VALIDATION_NON_N, (~validation_pool["high_impact_conflict"]).sum()),
    random_state=123,
)
validation_sample = pd.concat([hi_sample, non_sample], ignore_index=True)

manual_cols = (
    [
        "row_id",
        "file_name",
        "high_impact_conflict",
        "language",
        "assignment_type",
        "major",
        "university",
        "year_grouping",
        "word_count",
        "combined_ai_rate_100",
        "detector_score_range",
        "detector_score_variance",
    ]
    + score_cols
    + ["file_path_resolved"]
)
manual_validation = validation_sample[manual_cols].copy()
manual_validation["manual_template_artifacts"] = ""
manual_validation["manual_mixed_writing_style"] = ""
manual_validation["manual_reference_heavy_text"] = ""
manual_validation["manual_formulaic_structure"] = ""
manual_validation["manual_non_native_english_signals"] = ""
manual_validation["manual_technical_jargon"] = ""
manual_validation["manual_paraphrase_like_phrasing"] = ""
manual_validation["manual_formatting_artifacts"] = ""
manual_validation["manual_notes_no_authorship_claims"] = ""

manual_path = OUTPUT_DIR / "manual_validation_sample_high_conflict_features.csv"
manual_validation.to_csv(manual_path, index=False, encoding="utf-8-sig")
print("Saved manual validation sheet:", manual_path)
display(manual_validation)

In [ ]:
# Check for a completed manual validation CSV and, if present, compute per-feature
# agreement between the manual labels and the OpenAI-derived feature codes.
# Assumptions: manual file uses 'row_id' to join; manual columns for boolean
# labels may be textual ("true", "1", "yes", "false", "0", "no").
# Limitation: non-matching or missing columns are skipped; percent_agreement is
# reported only over rows where both manual and OpenAI values are present.
completed_manual_path = (
    OUTPUT_DIR / "manual_validation_sample_high_conflict_features_COMPLETED.csv"
)
if completed_manual_path.exists() and not feature_codes.empty:
    manual_done = pd.read_csv(completed_manual_path)
    merged_val = manual_done.merge(
        feature_codes, on="row_id", how="left", suffixes=("_manual", "_openai")
    )
    feature_names = [
        "template_artifacts",
        "mixed_writing_style",
        "reference_heavy_text",
        "formulaic_structure",
        "non_native_english_signals",
        "technical_jargon",
        "paraphrase_like_phrasing",
        "formatting_artifacts",
    ]
    agree_rows = []
    for f in feature_names:
        mcol = f"manual_{f}"
        if mcol in merged_val.columns and f in merged_val.columns:
            m = (
                merged_val[mcol]
                .astype(str)
                .str.lower()
                .map(
                    {
                        "true": True,
                        "1": True,
                        "yes": True,
                        "false": False,
                        "0": False,
                        "no": False,
                    }
                )
            )
            o = merged_val[f]
            valid = m.notna() & o.notna()
            agree_rows.append(
                {
                    "feature": f,
                    "n_valid": int(valid.sum()),
                    "percent_agreement": (
                        float((m[valid] == o[valid]).mean() * 100)
                        if valid.sum()
                        else np.nan
                    ),
                }
            )
    validation_agreement = pd.DataFrame(agree_rows)
    validation_agreement.to_csv(
        TABLE_DIR / "manual_openai_feature_validation_agreement.csv",
        index=False,
        encoding="utf-8-sig",
    )
    display(validation_agreement)
else:
    print(
        "No completed manual validation sheet found, or OpenAI feature codes are empty. This is expected before manual review."
    )

## 9. APA-ready tables and interpretation

This section writes concise APA-style tables and a cautious manuscript-ready interpretation. The interpretation uses descriptive evidence only and avoids any individual-authorship claims.

In [ ]:
def format_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "< .001"
    return f"{p:.3f}".lstrip("0")


def round_cols(df_in, digits=2):
    out = df_in.copy()
    for column in out.columns:
        if pd.api.types.is_float_dtype(out[column]):
            out[column] = out[column].round(digits)
    return out


# High-conflict distribution by grouping variable.
cat_table_rows = []
for variable, table in cat_crosstabs.items():
    for _, row in table.iterrows():
        cat_table_rows.append(
            {
                "Variable": variable.replace("_", " ").title(),
                "Category": row[variable],
                "Non-high-conflict n": int(row["non_high_conflict_n"]),
                "High-conflict n": int(row["high_conflict_n"]),
                "High-conflict % within category": row[
                    "high_conflict_percent_within_category"
                ],
            }
        )
apa_cat_table = pd.DataFrame(cat_table_rows)

apa_cont_table = continuous_comparison[
    [
        "variable",
        "high_conflict_mean",
        "high_conflict_sd",
        "high_conflict_median",
        "non_high_conflict_mean",
        "non_high_conflict_sd",
        "non_high_conflict_median",
        "mean_difference_high_minus_non",
        "mann_whitney_p",
        "cliffs_delta",
    ]
].copy()
apa_cont_table["mann_whitney_p"] = apa_cont_table["mann_whitney_p"].apply(format_p)
apa_cont_table = apa_cont_table.rename(
    columns={
        "variable": "Variable",
        "high_conflict_mean": "High-conflict M",
        "high_conflict_sd": "High-conflict SD",
        "high_conflict_median": "High-conflict median",
        "non_high_conflict_mean": "Non-high-conflict M",
        "non_high_conflict_sd": "Non-high-conflict SD",
        "non_high_conflict_median": "Non-high-conflict median",
        "mean_difference_high_minus_non": "Mean difference",
        "mann_whitney_p": "Mann-Whitney p",
        "cliffs_delta": "Cliff's delta",
    }
)

apa_tool_table = tool_participation.copy()
apa_tool_table["percent_le_30"] = apa_tool_table["percent_le_30"].round(1)
apa_tool_table["percent_ge_70"] = apa_tool_table["percent_ge_70"].round(1)
apa_tool_table["mean_score"] = apa_tool_table["mean_score"].round(2)
apa_tool_table["median_score"] = apa_tool_table["median_score"].round(2)

apa_pattern_table = pattern_counts.head(12).copy()

if not feature_codes.empty and "coding_error" not in feature_codes.columns:
    bool_features = [
        column
        for column in feature_codes.columns
        if column != "high_impact_conflict"
        and not feature_codes[column].dropna().empty
        and feature_codes[column].dropna().isin([True, False]).all()
    ]
    feature_rows = []
    for feature in bool_features:
        for group_value, group_label in [
            (True, "High-conflict"),
            (False, "Non-high-conflict"),
        ]:
            subset = feature_codes[feature_codes["high_impact_conflict"] == group_value]
            if len(subset):
                feature_rows.append(
                    {
                        "Feature": feature,
                        "Group": group_label,
                        "n": int(subset[feature].sum()),
                        "Percent": float(subset[feature].mean() * 100),
                    }
                )
    apa_feature_table = pd.DataFrame(feature_rows)
else:
    apa_feature_table = pd.DataFrame()

excel_path = OUTPUT_DIR / "APA_ready_high_impact_conflict_tables.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    round_cols(apa_cat_table, 1).to_excel(
        writer, sheet_name="Table1_categorical", index=False
    )
    round_cols(apa_cont_table, 2).to_excel(
        writer, sheet_name="Table2_continuous", index=False
    )
    apa_tool_table.to_excel(writer, sheet_name="Table3_tools", index=False)
    apa_pattern_table.to_excel(writer, sheet_name="Table4_patterns", index=False)
    if not apa_feature_table.empty:
        round_cols(apa_feature_table, 1).to_excel(
            writer, sheet_name="Table5_features", index=False
        )

apa_cat_table.to_csv(
    TABLE_DIR / "APA_Table1_categorical_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)
apa_cont_table.to_csv(
    TABLE_DIR / "APA_Table2_continuous_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)
apa_tool_table.to_csv(
    TABLE_DIR / "APA_Table3_detector_specific_patterns.csv",
    index=False,
    encoding="utf-8-sig",
)
apa_pattern_table.to_csv(
    TABLE_DIR / "APA_Table4_low_high_patterns.csv", index=False, encoding="utf-8-sig"
)

print("Saved APA-ready Excel workbook:", excel_path)

In [ ]:
# =========================================================
# 12. Generate cautious manuscript-ready interpretation
# =========================================================
# Determine strongest categorical differences by high-conflict percent.
top_category_rows = []
for var, tab in cat_crosstabs.items():
    temp = tab.copy()
    temp = temp[temp["total_n"] >= 10].sort_values(
        "high_conflict_percent_within_category", ascending=False
    )
    if len(temp):
        top = temp.iloc[0]
        top_category_rows.append(
            f"{var.replace('_', ' ')} was highest for {top[var]} ({top['high_conflict_percent_within_category']:.1f}%; {int(top['high_conflict_n'])}/{int(top['total_n'])})"
        )

overall_high_n = int(df["high_impact_conflict"].sum())
overall_n = len(df)
overall_pct = 100 * overall_high_n / overall_n
range_hi = df.loc[df["high_impact_conflict"], "detector_score_range"].mean()
range_non = df.loc[~df["high_impact_conflict"], "detector_score_range"].mean()
wc_hi = df.loc[df["high_impact_conflict"], "word_count"].median()
wc_non = df.loc[~df["high_impact_conflict"], "word_count"].median()

interpretation = f"""
Using the predefined high-impact conflict rule, {overall_high_n} of {overall_n} files ({overall_pct:.1f}%) were flagged as cases in which detector disagreement was most likely to affect interpretation. These files had much larger detector-score ranges than other files (mean range = {range_hi:.2f} percentage points vs. {range_non:.2f} percentage points), which confirms that the flag primarily captured disagreement rather than merely moderate combined scores. The distribution of high-impact conflicts was not uniform across the corpus; {('; '.join(top_category_rows[:5]))}. Median word count was {wc_hi:.0f} among high-impact conflicts and {wc_non:.0f} among other files. These patterns should be interpreted as corpus-level indicators of where detectors were least consistent. They do not show that any individual file was AI-generated, and they do not establish causality. Because language, assignment type, major, university, and year were partially overlapping in this corpus, apparent group differences may reflect bundled assignment contexts rather than separable effects of a single factor.
""".strip()

interpretation_path = OUTPUT_DIR / "manuscript_ready_high_conflict_interpretation.txt"
interpretation_path.write_text(interpretation, encoding="utf-8")
print(interpretation)
print("\nSaved:", interpretation_path)

In [ ]:
# Optional DOCX export: write APA-style tables summarizing corpus-level detector disagreements.
# Requires python-docx; skipped when DOCX_AVAILABLE is False. Output is descriptive and
# must not be used to label or classify individual files as AI-generated.
if DOCX_AVAILABLE:
    doc = Document()
    doc.add_heading("Paper 2 High-Impact Detector-Conflict Analysis", 0)
    doc.add_paragraph(
        "This report summarizes corpus-level detector-disagreement patterns. It does not classify any individual file as AI-generated."
    )

    doc.add_heading(
        "Table 1. Distribution of high-impact conflicts by grouping variable", level=1
    )
    t = doc.add_table(rows=1, cols=len(apa_cat_table.columns))
    t.style = "Table Grid"
    for j, col in enumerate(apa_cat_table.columns):
        t.rows[0].cells[j].text = str(col)
    for _, row in round_cols(apa_cat_table, 1).iterrows():
        cells = t.add_row().cells
        for j, col in enumerate(apa_cat_table.columns):
            cells[j].text = str(row[col])

    doc.add_heading("Table 2. Continuous comparisons", level=1)
    t = doc.add_table(rows=1, cols=len(apa_cont_table.columns))
    t.style = "Table Grid"
    for j, col in enumerate(apa_cont_table.columns):
        t.rows[0].cells[j].text = str(col)
    for _, row in round_cols(apa_cont_table, 2).iterrows():
        cells = t.add_row().cells
        for j, col in enumerate(apa_cont_table.columns):
            cells[j].text = str(row[col])

    doc.add_heading("Table 3. Detector-specific low/high participation", level=1)
    t = doc.add_table(rows=1, cols=len(apa_tool_table.columns))
    t.style = "Table Grid"
    for j, col in enumerate(apa_tool_table.columns):
        t.rows[0].cells[j].text = str(col)
    for _, row in apa_tool_table.iterrows():
        cells = t.add_row().cells
        for j, col in enumerate(apa_tool_table.columns):
            cells[j].text = str(row[col])

    doc.add_heading("Cautious manuscript-ready interpretation", level=1)
    doc.add_paragraph(interpretation)
    doc.add_paragraph(
        "Note. All findings are descriptive, corpus-level associations. They must not be used as evidence that any individual file was AI-generated."
    )

    docx_path = OUTPUT_DIR / "Paper2_High_Impact_Conflict_Analysis_APA_Tables.docx"
    doc.save(docx_path)
    print("Saved DOCX report:", docx_path)
else:
    print("python-docx not available; skipped DOCX report.")

## 10. Recommended manuscript wording

Use language like this when integrating the results:

> A high-impact conflict flag was used to identify files where detector disagreement was most likely to affect interpretation. A file met this definition when at least one detector scored it at ≤30% AI, at least one detector scored it at ≥70% AI, and the Combined AI rate fell between 40% and 60%. These files were analyzed as corpus-level disagreement cases only; they were not treated as evidence that any individual file was or was not AI-generated.

After running the notebook, paste the generated paragraph from `manuscript_ready_high_conflict_interpretation.txt` into the Results or supplementary appendix and revise it to match the final tables.